In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [3]:
dataset=pd.read_csv('Social_Network_Ads.csv')

In [4]:
dataset

,User ID,Gender,Age,EstimatedSalary,Purchased
0,15624510,Male,19,19000,0
1,15810944,Male,35,20000,0
2,15668575,Female,26,43000,0
3,15603246,Female,27,57000,0
4,15804002,Male,19,76000,0
...,...,...,...,...,...
395,15691863,Female,46,41000,1
396,15706071,Male,51,23000,1
397,15654296,Female,50,20000,1
398,15755018,Male,36,33000,0


In [5]:
dataset=pd.get_dummies(dataset,dtype=int,drop_first=True)

In [6]:
dataset

,User ID,Age,EstimatedSalary,Purchased,Gender_Male
0,15624510,19,19000,0,1
1,15810944,35,20000,0,1
2,15668575,26,43000,0,0
3,15603246,27,57000,0,0
4,15804002,19,76000,0,1
...,...,...,...,...,...
395,15691863,46,41000,1,0
396,15706071,51,23000,1,1
397,15654296,50,20000,1,0
398,15755018,36,33000,0,1


In [7]:
dataset.columns

Index(['User ID', 'Age', 'EstimatedSalary', 'Purchased', 'Gender_Male'], dtype='object')

In [8]:
independent=dataset[['Age', 'EstimatedSalary','Gender_Male']]

In [9]:
independent

,Age,EstimatedSalary,Gender_Male
0,19,19000,1
1,35,20000,1
2,26,43000,0
3,27,57000,0
4,19,76000,1
...,...,...,...
395,46,41000,0
396,51,23000,1
397,50,20000,0
398,36,33000,1


In [10]:
dependent=dataset['Purchased']

In [11]:
dependent

0      0
1      0
2      0
3      0
4      0
      ..
395    1
396    1
397    1
398    0
399    1
Name: Purchased, Length: 400, dtype: int64

In [12]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(independent, dependent, test_size = 1/3, random_state = 0)

In [13]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [16]:
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC
param_grid={ "kernel": ["poly"],
    "C": [0.1, 1, 10],
    "degree": [2, 3, 4],
    "gamma": ["scale", 0.01, 0.1],
    "coef0": [0.0, 0.5, 1.0]}
grid=GridSearchCV(SVC(),param_grid, refit=True, verbose=3, n_jobs=-1,scoring='f1_weighted')
grid.fit(X_train,y_train)

Fitting 5 folds for each of 81 candidates, totalling 405 fits


,estimator,SVC()
,param_grid,"{'C': [0.1, 1, ...], 'coef0': [0.0, 0.5, ...], 'degree': [2, 3, ...], 'gamma': ['scale', 0.01, ...], ...}"
,scoring,'f1_weighted'
,n_jobs,-1
,refit,True
,cv,None
,verbose,3
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,C,10


In [17]:
re=grid.cv_results_

In [18]:
re

{'mean_fit_time': array([0.00505114, 0.00468283, 0.00438242, 0.00415535, 0.00426822,
        0.00402865, 0.00376701, 0.00381622, 0.0041472 , 0.00381188,
        0.00360026, 0.00438004, 0.00415277, 0.00415335, 0.00411654,
        0.00379391, 0.00445971, 0.00323248, 0.00297856, 0.00411282,
        0.00419521, 0.00401073, 0.00442162, 0.00390697, 0.00331054,
        0.00452127, 0.00351377, 0.00466671, 0.00510406, 0.00374031,
        0.00384512, 0.00402899, 0.00418577, 0.00405173, 0.00469871,
        0.00399914, 0.00344191, 0.00431471, 0.00358734, 0.00349154,
        0.00399485, 0.0036612 , 0.0031662 , 0.00416188, 0.00314775,
        0.00298533, 0.003371  , 0.00363798, 0.00371685, 0.00375276,
        0.0033968 , 0.00405865, 0.00371399, 0.00349646, 0.00519238,
        0.00413041, 0.00394011, 0.00547686, 0.00422573, 0.00366735,
        0.00915565, 0.00378799, 0.00412035, 0.004142  , 0.00379162,
        0.00423608, 0.0044981 , 0.00368872, 0.00353994, 0.00519676,
        0.00423746, 0.00371561,

In [19]:
grid_predictions = grid.predict(X_test) 
   

from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, grid_predictions)



# print classification report 
from sklearn.metrics import classification_report
clf_report = classification_report(y_test, grid_predictions)

In [20]:
from sklearn.metrics import f1_score
f1_macro=f1_score(y_test,grid_predictions,average='weighted')
print("The f1_macro value for best parameter {}:".format(grid.best_params_),f1_macro)

The f1_macro value for best parameter {'C': 10, 'coef0': 1.0, 'degree': 3, 'gamma': 0.1, 'kernel': 'poly'}: 0.9183922682195829


In [21]:
print("The confusion Matrix:\n",cm)

The confusion Matrix:
 [[78  7]
 [ 4 45]]


In [22]:
print("The report:\n",clf_report)

The report:
               precision    recall  f1-score   support

           0       0.95      0.92      0.93        85
           1       0.87      0.92      0.89        49

    accuracy                           0.92       134
   macro avg       0.91      0.92      0.91       134
weighted avg       0.92      0.92      0.92       134



In [24]:
table=pd.DataFrame.from_dict(re)

In [25]:
table

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_C,param_coef0,param_degree,param_gamma,param_kernel,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.005051,0.000735,0.006177,0.000594,0.1,0.0,2,scale,poly,"{'C': 0.1, 'coef0': 0.0, 'degree': 2, 'gamma':...",0.715520,0.776838,0.617764,0.747667,0.772377,0.726033,0.058360,56
1,0.004683,0.001029,0.005573,0.001245,0.1,0.0,2,0.01,poly,"{'C': 0.1, 'coef0': 0.0, 'degree': 2, 'gamma':...",0.509779,0.525300,0.501410,0.501410,0.501410,0.507862,0.009302,64
2,0.004382,0.001306,0.005705,0.001042,0.1,0.0,2,0.1,poly,"{'C': 0.1, 'coef0': 0.0, 'degree': 2, 'gamma':...",0.509779,0.525300,0.501410,0.501410,0.501410,0.507862,0.009302,64
3,0.004155,0.000784,0.005410,0.000514,0.1,0.0,3,scale,poly,"{'C': 0.1, 'coef0': 0.0, 'degree': 3, 'gamma':...",0.768633,0.727808,0.621589,0.862442,0.789723,0.754039,0.079333,52
4,0.004268,0.000648,0.004762,0.000793,0.1,0.0,3,0.01,poly,"{'C': 0.1, 'coef0': 0.0, 'degree': 3, 'gamma':...",0.509779,0.525300,0.501410,0.501410,0.501410,0.507862,0.009302,64
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76,0.004111,0.000745,0.005224,0.000470,10.0,1.0,3,0.01,poly,"{'C': 10, 'coef0': 1.0, 'degree': 3, 'gamma': ...",0.799620,0.811321,0.727648,0.943041,0.922185,0.840763,0.080554,34
77,0.003327,0.000416,0.005473,0.000949,10.0,1.0,3,0.1,poly,"{'C': 10, 'coef0': 1.0, 'degree': 3, 'gamma': ...",0.867478,0.886792,0.850543,0.962636,0.943041,0.902098,0.043432,1
78,0.006288,0.001029,0.005153,0.000421,10.0,1.0,4,scale,poly,"{'C': 10, 'coef0': 1.0, 'degree': 4, 'gamma': ...",0.867478,0.866968,0.869709,0.925272,0.903610,0.886607,0.023754,20
79,0.004045,0.000643,0.005256,0.000835,10.0,1.0,4,0.01,poly,"{'C': 10, 'coef0': 1.0, 'degree': 4, 'gamma': ...",0.799620,0.831253,0.749387,0.943041,0.981014,0.860863,0.087457,31
